<a href="https://colab.research.google.com/github/diegozuu/asistente-para-planificacion-de-toma-de-ramos/blob/main/PROTOTIPO_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. INSTALACION E IMPORTACION DE LIBRERIAS
**¿Que hace esta celda?**
Prepara nuestro entorno de trabajo. Instala el motor Tesseract (para que la IA lea imagenes) y carga las librerias necesarias como pandas para el Excel, gradio para la interfaz visual y pytesseract para el Reconocimiento Optico de Caracteres (OCR).

In [1]:
!apt-get install -y tesseract-ocr tesseract-ocr-spa -q
!pip install gradio openai pandas pytesseract Pillow -q

import pandas as pd
import gradio as gr
import json
from openai import OpenAI
from google.colab import userdata
import pytesseract
from PIL import Image

print("[Exito] Librerias importadas correctamente con soporte para imagenes (OCR)")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (5.3.4-1build5).
The following NEW packages will be installed:
  tesseract-ocr-spa
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 1,065 kB of archives.
After this operation, 2,309 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 tesseract-ocr-spa all 1:4.1.0-2 [1,065 kB]
Fetched 1,065 kB in 0s (2,734 kB/s)
Selecting previously unselected package tesseract-ocr-spa.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-spa_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-spa (1:4.1.0-2) ...
Setting up tesseract-ocr-spa (1:4.1.0-2) ...
[Exito] Librerias importadas correctamente con soporte para imagenes (OCR)


# 2. CONEXION A LA API Y CARGA DEL EXCEL
**¿Que hace esta celda?**
Se conecta al cerebro de nuestra Inteligencia Artificial usando la clave de Groq/OpenAI. Ademas, lee el archivo Excel y lo guarda en una tabla virtual (DataFrame) para que el bot pueda buscar los horarios alli.

In [2]:
try:
    cliente_ai = OpenAI(
        api_key=userdata.get('OPENAI_API_KEY'),
        base_url="https://api.groq.com/openai/v1"
    )
    modelo_elegido = "openai/gpt-oss-120b"
    print("[Exito] API key y modelo IA funcionando correctamente")
except Exception as e:
    print(f"[Error] Al conectar con la API: {e}")

try:
    df_horarios = pd.read_excel('SAN BERNARDO 2026 002.xlsx', sheet_name='BASE')
    print("[Exito] Excel de horarios cargado correctamente.")
except Exception as e:
    print(f"[Error] Al cargar el Excel: {e}")
    df_horarios = pd.DataFrame()

[Exito] API key y modelo IA funcionando correctamente
[Exito] Excel de horarios cargado correctamente.


# 3. FUNCION DE EXTRACCION DE TEXTO (OCR)
**¿Que hace esta celda?**
Crea una herramienta personalizada. Su unico trabajo es tomar la ruta de una imagen (.png o .jpg), abrirla, escanearla usando pytesseract en español, y devolver todo el texto que encontro escrito en ella.

In [3]:
def extraer_texto_imagen(ruta_imagen):
    texto = ""
    try:
        print(f"Aplicando OCR a la imagen: {ruta_imagen}...")
        imagen = Image.open(ruta_imagen)
        texto = pytesseract.image_to_string(imagen, lang='spa')
    except Exception as e:
        print(f"[Error] Procesando {ruta_imagen}: {e}")
    return texto

# 4. PROCESAMIENTO DE LAS MALLAS (IMAGENES)
**¿Que hace esta celda?**
Toma todas las fotos de las mallas curriculares que subiste y usa la herramienta del paso anterior para extraer y guardar todo su texto en la memoria de Colab, asociandolo a palabras clave como 'Mecanica' o 'Desarrollo'.

In [5]:
mallas_archivos = {
    "Mecánica": "MALLA_CURRICULAR-Ingeniería-en-Mecánica-Automotriz-y-Autotrónica-DuocUC-2026.jpg",
    "Finanzas": "MALLA_CURRICULAR-Ingeniería-en-Administración-mención-Finanzas-DuocUC-2026.jpg",
    "Automatización": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Automatización-DuocUC-2026.jpg",
    "Ciencia de Datos": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ciencia-de-Datos-DuocUC-2026.jpg",
    "Desarrollo": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Desarrollo-de-Software-DuocUC-2026.jpg",
    "Ingeniería de Datos": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ingeniería-de-Datos-DuocUC-2026.jpg",
    "Inteligencia Artificial": "MALLA_CURRICULAR-Ingeniería-en-Informática-Mención-Inteligencia-Artificial-DuocUC-2026.jpg"
}

try:
    base_mallas_txt = {clave: extraer_texto_imagen(arch) for clave, arch in mallas_archivos.items()}
    print("\n[Exito] Mallas curriculares en imagen procesadas con exito.")
except Exception as e:
    print(f"[Error] Al procesar las imagenes: {e}")

Aplicando OCR a la imagen: MALLA_CURRICULAR-Ingeniería-en-Mecánica-Automotriz-y-Autotrónica-DuocUC-2026.jpg...
Aplicando OCR a la imagen: MALLA_CURRICULAR-Ingeniería-en-Administración-mención-Finanzas-DuocUC-2026.jpg...
Aplicando OCR a la imagen: MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Automatización-DuocUC-2026.jpg...
Aplicando OCR a la imagen: MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ciencia-de-Datos-DuocUC-2026.jpg...
Aplicando OCR a la imagen: MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Desarrollo-de-Software-DuocUC-2026.jpg...
Aplicando OCR a la imagen: MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ingeniería-de-Datos-DuocUC-2026.jpg...
Aplicando OCR a la imagen: MALLA_CURRICULAR-Ingeniería-en-Informática-Mención-Inteligencia-Artificial-DuocUC-2026.jpg...

[Exito] Mallas curriculares en imagen procesadas con exito.


# 5. SISTEMA MULTI-AGENTE CON MEMORIA DE ESTADO Y SEGUIMIENTO
**¿Que hace esta celda?**
Aplica la materia de 'Agentes con Memoria' utilizando una variable de estado que recuerda la conversacion.
- **Memoria de Llenado:** Si das un dato parcial (solo semestre), el Agente 1 lo guarda y te pide los restantes.
- **Memoria de Seguimiento:** Si ya te dio tu horario y luego preguntas '¿y en vespertino?', el sistema recuerda tu carrera y semestre anteriores, actualiza solo la jornada y genera un horario nuevo.

In [6]:
def agente_analista(mensaje_usuario, estado_actual):
    """Agente 1: Analiza el texto y actualiza la memoria del estado"""
    prompt = f"""
    Eres un analizador de texto estricto.

    ESTADO ACTUAL EN MEMORIA:
    {json.dumps(estado_actual)}

    NUEVO MENSAJE DEL USUARIO:
    "{mensaje_usuario}"

    REGLAS:
    1. Busca referencias a la carrera, semestre y jornada (Diurna o Vespertina) en el nuevo mensaje.
    2. Si el usuario menciona un dato nuevo, agregalo al estado.
    3. Si el usuario pide cambiar un dato existente (ej. "y en vespertino?"), reemplazalo.
    4. Si el mensaje no cambia un dato, manten el valor que ya estaba en el ESTADO ACTUAL.
    5. Responde UNICAMENTE con un objeto JSON valido con las claves "carrera", "semestre" y "jornada". No uses texto adicional ni explicaciones.
    """
    try:
        respuesta = cliente_ai.chat.completions.create(
            model=modelo_elegido,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        contenido = respuesta.choices[0].message.content.replace("```json", "").replace("```", "").strip()
        nuevo_estado = json.loads(contenido)
        return nuevo_estado
    except Exception as e:
        print("Error en Agente Analista:", e)
        return estado_actual

def procesar_solicitud(mensaje, estado_actual):
    """Funcion principal que dirige el flujo con memoria"""
    if not mensaje.strip():
        return "Por favor, ingresa tu solicitud.", estado_actual

    # 1. El Agente 1 lee el mensaje y actualiza la memoria
    nuevo_estado = agente_analista(mensaje, estado_actual)

    carrera = nuevo_estado.get("carrera")
    semestre = nuevo_estado.get("semestre")
    jornada = nuevo_estado.get("jornada")

    # 2. Verificamos si la memoria esta completa
    faltantes = []
    if not carrera: faltantes.append("tu carrera")
    if not semestre: faltantes.append("el semestre a cursar")
    if not jornada: faltantes.append("tu jornada (Diurna o Vespertina)")

    # Si faltan datos, aplicamos Memoria de Llenado
    if faltantes:
        recibidos = []
        if carrera: recibidos.append(f"Carrera: {carrera}")
        if semestre: recibidos.append(f"Semestre: {semestre}")
        if jornada: recibidos.append(f"Jornada: {jornada}")

        msg = "Datos registrados en memoria hasta el momento:\n"
        if recibidos:
            for item in recibidos:
                msg += f"- {item}\n"
            msg += "\n"
        else:
            msg += "- Ningun dato registrado aun.\n\n"

        msg += f"Para proceder con la generacion del boletin, por favor indicame: {', '.join(faltantes)}."
        return msg, nuevo_estado

    # 3. Si la memoria esta completa, aplicamos el Motor de Busqueda
    texto_malla = ""
    horarios_txt = ""
    for clave in mallas_archivos.keys():
        if clave.lower() in str(carrera).lower():
            texto_malla = base_mallas_txt.get(clave, "")
            filtro = df_horarios['Carrera'].str.contains(clave, case=False, na=False)
            cols = ['Carrera', 'Nombre Asignatura', 'Sección', 'Horario', 'Docente']
            df_sub = df_horarios[filtro][cols]
            horarios_txt = df_sub.to_string(index=False)
            break

    # 4. Agente 2: Generador de Respuesta
    prompt_agente_2 = f"""
    Eres el Asistente Virtual DUOC UC.
    Redacta la propuesta de horario en base a la memoria del sistema:
    Alumno de {carrera}, {semestre} semestre, jornada {jornada}.

    DATOS:
    Malla extraida: {texto_malla if texto_malla else 'Sin datos'}
    Excel: {horarios_txt if horarios_txt else 'Sin datos'}

    INSTRUCCION ESTRICTA: Crea una tabla en Markdown estricto. NO utilices emojis en ninguna parte de tu respuesta.
    Formato:
    | Modulo Horario | Lunes | Martes | Miercoles | Jueves | Viernes | Sabado |
    |---|---|---|---|---|---|---|
    | 08:30 - 09:50 | RAMO<br>SECCION | | | | | |
    """
    try:
        respuesta = cliente_ai.chat.completions.create(
            model=modelo_elegido,
            messages=[{"role": "user", "content": prompt_agente_2}],
            temperature=0.3
        )
        return respuesta.choices[0].message.content, nuevo_estado
    except Exception as e:
        return f"Error en Agente 2: {e}", nuevo_estado

# 6. ESTILOS VISUALES (HTML Y CSS)
**¿Que hace esta celda?**
Define el diseño de la pagina. Elimina emojis y mantiene la estructura de 'caja negra' para que el boletin academico destaquen los colores corporativos (Fondo oscuro, texto blanco, bordes amarillos).

In [7]:
estilos_css = """
:root {
    --duoc-yellow: #FFB81C;
    --duoc-black: #1A1A1A;
    --duoc-white: #FFFFFF;
}
body, .gradio-container {
    background-color: #EAECEF !important;
    font-family: 'Arial', sans-serif !important;
}
.gradio-container h1, .gradio-container h2, .gradio-container h3, .gradio-container p {
    color: var(--duoc-black) !important;
}
.encabezado-duoc {
    background-color: var(--duoc-black);
    padding: 25px;
    border-bottom: 5px solid var(--duoc-yellow);
    display: flex; align-items: center; justify-content: center;
    border-radius: 12px 12px 0 0;
}
.encabezado-duoc h1 { color: var(--duoc-yellow) !important; margin: 0; font-size: 28px; }
.encabezado-duoc p { color: #e0e0e0 !important; margin: 5px 0 0 0; }

.caja-negra {
    background-color: var(--duoc-black) !important;
    color: var(--duoc-white) !important;
    padding: 20px !important;
    border-radius: 10px !important;
    border: 2px solid var(--duoc-yellow) !important;
    box-shadow: 0px 4px 10px rgba(0,0,0,0.3);
}
.caja-negra h1, .caja-negra h2, .caja-negra h3, .caja-negra p, .caja-negra li, .caja-negra span {
    color: var(--duoc-white) !important;
}
.caja-negra table {
    background-color: #2D2D2D !important;
    color: var(--duoc-white) !important;
    border-color: #444 !important;
}
.caja-negra th {
    background-color: #000000 !important;
    color: var(--duoc-yellow) !important;
    text-align: center !important;
}
.caja-negra td {
    background-color: #2D2D2D !important;
    color: var(--duoc-white) !important;
    border-color: #444 !important;
    text-align: center !important;
}
.caja-negra td * { color: var(--duoc-white) !important; }
"""

html_cabecera = """
<div class="encabezado-duoc">
    <div style="text-align: center;">
        <h1>Portal Academico de Estudiantes</h1>
        <p>Buscador Inteligente de Horarios y Boletin Academico</p>
    </div>
</div>
"""

# 7. INTERFAZ GRAFICA CON MEMORIA DE ESTADO
**¿Que hace esta celda?**
Construye la pagina web final. Aqui se introduce el componente 'gr.State', que actua como el cerebro a corto plazo del simulador, almacenando la carrera, semestre y jornada entre cada interaccion del usuario.

In [9]:
with gr.Blocks(css=estilos_css) as portal_simulado:
    gr.HTML(html_cabecera)

    # Variable invisible que guarda la memoria de la sesion
    estado_memoria = gr.State({"carrera": None, "semestre": None, "jornada": None})

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Ingresa tu solicitud")
            entrada_usuario = gr.Textbox(
                label="",
                placeholder="Ej: Soy de Desarrollo, paso a 4to semestre en jornada Diurna.",
                lines=4,
                container=False
            )
            boton_generar = gr.Button("Enviar / Actualizar", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### Tu Boletin Academico")
            # Caja Negra
            salida_bot = gr.Markdown(
                "La informacion de tu planificacion aparecera aqui.",
                elem_classes=["caja-negra"]
            )

    # Conectamos las funciones considerando la entrada, salida y la memoria actual
    boton_generar.click(
        fn=procesar_solicitud,
        inputs=[entrada_usuario, estado_memoria],
        outputs=[salida_bot, estado_memoria]
    )



/tmp/ipykernel_1124/3269470597.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=estilos_css) as portal_simulado:


In [10]:
print("Generando link del html en funcionamiento....")
portal_simulado.launch(share=True)

Generando link del html en funcionamiento....
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fbd75ba072ad248453.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
